
#### 10 — Transformer Foundations

##### 1. Purpose

This notebook builds the conceptual and practical foundation required to understand Transformer-based NLP models.

Previous notebooks progressed through:

- Bag of Words
- TF-IDF
- Logistic Regression
- Pretrained sentence embeddings
- Neural Networks

This notebook opens the pretrained Transformer used earlier and examines how text moves through the Transformer pipeline.

The notebook focuses on:

- tokenization
- subword tokens
- token IDs
- token embeddings
- positional information
- self-attention
- Query, Key, and Value
- scaled dot-product attention
- multi-head attention
- contextual token representations
- padding
- attention masks
- sentence pooling
- encoder vs decoder Transformers
- pretraining vs fine-tuning

No ticket classifier is trained in this notebook.

The next notebook will use these foundations to build a Transformer-based support-ticket classifier.

##### 2. Technologies

- Python
- PyTorch
- Hugging Face Transformers
- Sentence Transformers
- NumPy
- Pandas
- Databricks

##### 3. Input and Output

##### Input

A small set of support-ticket text examples and the pretrained model: `sentence-transformers/all-MiniLM-L6-v2`

The examples are intentionally small so that internal Transformer representations can be inspected clearly.

##### Output

This notebook demonstrates:

- tokenizer output
- token IDs
- attention masks
- padding
- Transformer architecture configuration
- hidden-state dimensions
- contextual token representations
- sentence embedding dimensions
- cosine similarity between contextual sentence embeddings

##### 4. Transformer Architecture

``` text

Text
↓ 
Tokenizer
↓
Tokens / Subwords
↓
Token IDs
↓  
Token Embeddings  
+  
Position Information
↓  
Position-aware Token Representations
↓  
Multi-Head Self-Attention
↓  
Contextual Token Representations
↓  
Feed-Forward Neural Network
↓  
Additional Transformer Layers
↓  
Final Contextual Token Representations
↓  
Pooling
↓  
Fixed-Size Sentence Embedding

```

##### 5. Import Libraries

In [0]:
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from src.project_config import (
    EMBEDDING_MODEL_NAME,
)

##### 6. Reproducibility

In [0]:
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

##### 7. Select Compute Device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {device}")

##### 8. Load the Pretrained Sentence Transformer

In [0]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=str(device),
)

print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME,
)

print(
    "Sentence embedding dimension:",
    embedding_model.get_sentence_embedding_dimension(),
)

384 = number of numerical features in every sentence embedding produced by this model

##### 9. Inspect the SentenceTransformer Components

In [0]:
#A SentenceTransformer is more than just a single operation.
print(embedding_model)

In [0]:
for index, module in enumerate(
    embedding_model._modules.values()
):
    print(
        f"Module {index}: "
        f"{module.__class__.__name__}"
    )

##### 10. Access the Tokenizer and Transformer

In [0]:
transformer_module = embedding_model[0]
tokenizer = transformer_module.tokenizer
transformer_model = transformer_module.auto_model

In [0]:
tokenizer

In [0]:
print(
    "Tokenizer:",
    tokenizer.__class__.__name__,
)

print(
    "Transformer model:",
    transformer_model.__class__.__name__,
)

##### 11. Inspect the Transformer Configuration

In [0]:
config = transformer_model.config

print(
    "Hidden size:",
    config.hidden_size,
)

print(
    "Transformer layers:",
    config.num_hidden_layers,
)

print(
    "Attention heads:",
    config.num_attention_heads,
)

print(
    "Maximum position embeddings:",
    config.max_position_embeddings,
)

print(
    "SentenceTransformer max sequence length:",
    embedding_model.max_seq_length,
)

hidden_size → number of numerical features representing each token internally

max_seq_length → number of token positions accepted by the SentenceTransformer pipeline

##### 12. Create Example Support Tickets

In [0]:
ticket_examples = [
    "account locked after many attempts",
    "cancel my account",
    "refund my monthly bill",
    "internet connection keeps dropping",
]

ticket_df = pd.DataFrame(
    {
        "ticket_text": ticket_examples
    }
)

display(ticket_df)

##### 13. Tokenization

In [0]:
example_text = (
    "account locked after many attempts"
)

tokens = tokenizer.tokenize(
    example_text
)

print(tokens)

##### 14. Tokens vs Token IDs

In [0]:
token_ids = tokenizer.convert_tokens_to_ids(
    tokens
)

token_table = pd.DataFrame(
    {
        "token": tokens,
        "token_id": token_ids,
    }
)

display(token_table)

##### 15. Special Tokens

In [0]:
encoded_single = tokenizer(
    example_text,
    return_tensors="pt",
)

print(
    encoded_single["input_ids"]
)

In [0]:
complete_token_ids = (
    encoded_single["input_ids"][0]
    .tolist()
)

complete_tokens = (
    tokenizer.convert_ids_to_tokens(
        complete_token_ids
    )
)

complete_token_table = pd.DataFrame(
    {
        "token": complete_tokens,
        "token_id": complete_token_ids,
    }
)

display(complete_token_table)

##### 16. Why Position Information Is Needed

Consider:

- dog bites man
- man bites dog

Both sentences contain exactly the same words.

A Bag-of-Words representation could produce the same counts.

But their meanings are very different.

The Transformer therefore needs information about token order.

Conceptually:

``` text

Token embedding
      +
Position information
      ↓
Position-aware representation

```

This provides information about:

WHAT token is this?
+
WHERE does it occur?

The exact implementation of positional information depends on the Transformer architecture.

##### 17. Inspect the Token Embedding Layer

In [0]:
print(
    transformer_model.embeddings
)

- For this BERT/MiniLM-style architecture, the embedding component includes learned structures associated with input representation.
- vocabulary_size × hidden_size
``` text

token ID
    ↓
embedding-table lookup
    ↓
hidden_size numerical values

```

In [0]:
word_embedding_layer = (
    transformer_model
    .embeddings
    .word_embeddings
)

print(
    "Token embedding table shape:",
    tuple(
        word_embedding_layer.weight.shape
    ),
)

In [0]:
transformer_model.embeddings.word_embeddings

##### 18. Retrieve a Real Token Embedding

In [0]:
account_token_id = (
    tokenizer.convert_tokens_to_ids(
        "account"
    )
)

print(
    "account token ID:",
    account_token_id,
)

In [0]:
with torch.no_grad():

    account_embedding = (
        word_embedding_layer.weight[
            account_token_id
        ]
    )

print(
    "Token embedding shape:",
    account_embedding.shape,
)

print(
    "First 10 values:"
)

print(
    account_embedding[:10]
)

- The shape should correspond to the model's hidden dimension.
- These numbers are learned parameters.
- They were not manually assigned.

##### 19. Where Did Those Token Embedding Numbers Come From?


During the model's original pretraining:

``` text

Initial parameters
↓
Forward propagation
↓
Training objective
↓
Loss
↓
Backpropagation
↓
Gradients
↓
Optimizer
↓
Parameter updates
↓
Repeat many times
↓
Learned pretrained parameters

```

Token embeddings are therefore trainable model parameters.

When we download the pretrained model, we receive those already-learned parameter values.

##### 20. Inspect Position Embeddings

In [0]:
position_embedding_layer = (
    transformer_model
    .embeddings
    .position_embeddings
)

print(
    "Position embedding table shape:",
    tuple(
        position_embedding_layer.weight.shape
    ),
)

``` text

position
   ↓
position embedding lookup
   ↓
hidden-size vector

```

In [0]:
with torch.no_grad():

    position_1_embedding = (
        position_embedding_layer.weight[1]
    )

print(
    "Position embedding shape:",
    position_1_embedding.shape,
)

print(
    "First 10 values:"
)

print(
    position_1_embedding[:10]
)

These are real learned parameters from the pretrained model.

##### 21. Token Embedding + Position Information


Conceptually:

``` text

"account"
    ↓
token ID
    ↓
token embedding
    ↓
384 values

Position
    ↓
position embedding
    ↓
384 values

        ↓ combine

position-aware
token representation

```

Token embedding → WHAT token?

Position information → WHERE?

Self-attention → What do surrounding tokens tell me?

Position information alone does not create full contextual meaning.

##### 22. Padding Problem

In [0]:
variable_length_tickets = [
    "refund",
    "account locked",
    (
        "customer cannot login "
        "because account is locked"
    ),
]

In [0]:
for text in variable_length_tickets:

    text_tokens = tokenizer.tokenize(
        text
    )

    print(
        len(text_tokens),
        text_tokens,
    )

- The sequences have different lengths.
- Neural-network batches need compatible tensor dimensions.

##### 23. Padding

In [0]:
encoded_batch = tokenizer(
    variable_length_tickets,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

In [0]:
print(
    "input_ids shape:",
    encoded_batch[
        "input_ids"
    ].shape,
)

print(
    encoded_batch[
        "input_ids"
    ]
)

Padding makes all sequences in the batch the same length.

Conceptually:

refund          PAD PAD PAD ...

account locked  PAD PAD PAD ...

longer ticket   ...

##### 24. Attention Mask

In [0]:
print(
    "attention_mask shape:",
    encoded_batch[
        "attention_mask"
    ].shape,
)

print(
    encoded_batch[
        "attention_mask"
    ]
)

Typically:

- 1 → real token
- 0 → padding position

Important:

attention_mask
≠
attention weights

The attention mask identifies valid token positions.

Actual attention weights are calculated later through self-attention.

##### 25. Visualize Tokens, IDs, and Attention Mask

In [0]:
#Let's inspect the first padded ticket.

row_index = 0

row_ids = (
    encoded_batch[
        "input_ids"
    ][row_index]
    .tolist()
)

row_mask = (
    encoded_batch[
        "attention_mask"
    ][row_index]
    .tolist()
)

row_tokens = (
    tokenizer.convert_ids_to_tokens(
        row_ids
    )
)

padding_table = pd.DataFrame(
    {
        "token": row_tokens,
        "token_id": row_ids,
        "attention_mask": row_mask,
    }
)

display(padding_table)

#This clearly shows which positions are actual tokens and which are padding.

##### 26. Padding Mask vs Causal Mask

##### Padding Attention Mask

Purpose: Ignore artificial padding positions.

Example:

refund | PAD | PAD

1      | 0   | 0


##### Causal Mask

Purpose: Prevent a decoder-style Transformer from seeing future tokens.

Example:

Token 1 → may see token 1

Token 2 → may see tokens 1–2

Token 3 → may see tokens 1–3


These masks solve different problems.

##### 27. Run the Transformer Directly

In [0]:
#Now let's send the tokenized batch through the underlying Transformer. Move inputs to the same compute device:

model_inputs = {
    key: value.to(device)
    for key, value
    in encoded_batch.items()
}

In [0]:
encoded_batch.items()

In [0]:
model_inputs

In [0]:
#Run inference:

transformer_model.eval()

with torch.no_grad():

    transformer_outputs = (
        transformer_model(
            **model_inputs
        )
    )

In [0]:
transformer_outputs


In [0]:
last_hidden_state = (
    transformer_outputs
    .last_hidden_state
)

print(
    "Last hidden state shape:",
    last_hidden_state.shape,
)

##### 28. Understand the Hidden-State Shap

Suppose:

- batch_size = 3
- sequence_length = 9
- hidden_size = 384

Then: last_hidden_state.shape = (3, 9, 384)

Meaning : 3 tickets × 9 token positions × 384 contextual features per token

This is very different from: sentence embedding shape = (3, 384)

##### 29. Inspect One Contextual Token Vector

In [0]:
first_ticket_first_token = (
    last_hidden_state[
        0,
        0,
        :
    ]
)

print(
    "Contextual token vector shape:",
    first_ticket_first_token.shape,
)

print(
    "First 10 values:"
)

print(
    first_ticket_first_token[:10]
)

- This representation has already passed through the Transformer layers.
- Therefore it is no longer merely the original token embedding.
- It is a contextual token representation.

##### 30. Self-Attention Concept

For each token, the Transformer computes:

Q = XW_Q

K = XW_K

V = XW_V

Where:

- W_Q is a learned Query projection
- W_K is a learned Key projection
- W_V is a learned Value projection

Q, K, and V themselves are calculated dynamically for the current input.

Attention is conceptually:

$$ Attention(Q,K,V) = softmax \left( \frac{QK^T} {\sqrt{d_k}} \right) V $$

Interpretation:

``` text

Q × Kᵀ  --> raw attention scores
↓
scale --> numerically manageable scores
↓
Softmax  --> attention weights (sum to 1 across allowed positions)
↓
× V -->  weighted combination of token information

↓
context-enriched representation

```

##### 31. Attention Scores vs Classification Logits

The pattern looks similar to classification but represents something different.

##### Classification

``` text

Raw class scores
↓
Softmax
↓
Class probabilities

```


##### Self-Attention

``` text

Raw token relevance scores
↓
Softmax
↓
Attention weights

```

Both use Softmax.

But:  class probability ≠ attention weight

##### 32. Query, Key, and Value

A useful conceptual interpretation:

Query --> "What information am I looking for?"


Key --> "What information do I contain that may be relevant?"


Value --> "What information should I contribute if I am relevant?"

These are conceptual explanations.

Mathematically, they are vectors generated using learned projection matrices.

##### 33. Learned Parameters vs Calculated Values


Learned and stored with the pretrained Transformer:

- token embedding parameters
- position-related parameters
- W_Q
- W_K
- W_V
- attention output projections
- feed-forward weights
- biases
- normalization parameters


Calculated dynamically for each input:

- Q
- K
- V
- attention scores
- attention weights
- contextual token representations

This distinction is essential.

##### 34. Multi-Head Attention



A single attention head represents one self-attention calculation.

``` text

Input X
   ↓
W_Q
W_K
W_V
   ↓
Q K V
   ↓
Self-Attention
   ↓
Head output

```

Multi-head attention performs several such calculations in parallel:

``` text

                  X

        ┌─────────┼─────────┐
        ↓         ↓         ↓

      Head 1    Head 2    Head 3
        ↓         ↓         ↓

      output    output    output

        └─────────┼─────────┘
                  ↓

             concatenate

                  ↓

          output projection

```

Different heads use different learned projections.

This allows multiple attention patterns to be learned from the same sequence.

##### 35. Inspect Number of Attention Heads

In [0]:

num_attention_heads = (
    config.num_attention_heads
)

hidden_size = (
    config.hidden_size
)

print(
    "Hidden size:",
    hidden_size,
)

print(
    "Attention heads:",
    num_attention_heads,
)

print(
    "Dimension per head:",
    hidden_size
    // num_attention_heads,
)

#This gives the actual dimensions for the loaded MiniLM model rather than relying on assumptions.

##### 36. Transformer Layers

In [0]:
print(
    "Number of Transformer layers:",
    config.num_hidden_layers,
)

Conceptually:

``` text

Input token representations
        ↓
Transformer Layer 1
        ↓
Transformer Layer 2
        ↓
...
        ↓
Final Transformer Layer
        ↓
Contextual token representations

```

For this MiniLM model, the architecture tells us how many encoder layers are present.

##### 37. Feed-Forward Neural Network

Attention is not the entire Transformer.

Each Transformer encoder layer also contains a feed-forward neural network.

Conceptually:
``` text

Multi-Head Attention
        ↓
contextual representation
        ↓
Feed-Forward Neural Network
        ↓
updated representation

```

This connects directly to our previous neural-network learning.

Transformers are neural networks that add a sophisticated attention architecture.

##### 38. Residual Connections



Conceptually:

``` text

X ─────────────────┐
│                  │
↓                  │
Transformation     │
│                  │
↓                  │
+ ←────────────────┘
↓
updated representation

```

Instead of forcing every layer to replace its input completely, the original representation is carried forward and combined with the transformation.

Residual connections help deep networks train effectively and preserve useful information.

##### 39. Layer Normalization

Layer normalization helps keep intermediate numerical representations well behaved during deep-network training.

At the conceptual level:

``` text

Transformer operation
↓
Layer normalization
↓
more stable representation

```

It is another trainable component of the Transformer architecture.

##### 40. Generate Sentence Embeddings

In [0]:
sentence_embeddings = (
    embedding_model.encode(
        variable_length_tickets,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
)

print(
    "Sentence embedding shape:",
    sentence_embeddings.shape,
)

3 tickets
×
384 features per ticket

##### 41. Token Representations vs Sentence Embeddings

Transformer output: batch × sequence length × 384

Example: (3, 9, 384)

          ↓
        pooling

SentenceTransformer output: batch × 384

Example: (3, 384)

``` text

many contextual token vectors
             ↓
           pooling
             ↓
one fixed-size sentence vector

```

##### 42. Pooling

For SentenceTransformer models, pooling converts token-level representations into one sentence-level representation.

Conceptually:

``` text

account   → 384 values
locked    → 384 values
after     → 384 values
many      → 384 values
attempts  → 384 values

            ↓

          pooling

            ↓

one ticket embedding
=
384 values

```

Padding tokens must not contribute like real tokens, which is one reason the attention mask is important to the pooling pipeline as well.

##### 43. Verify Sentence Embedding Dimension

In [0]:
for text, embedding in zip(
    variable_length_tickets,
    sentence_embeddings,
):

    print(
        f"{text!r}"
    )

    print(
        "Embedding dimension:",
        len(embedding),
    )

    print()

No matter whether the ticket contains one word or several words: embedding dimension = 384

The model architecture determines this output size.

##### 44. Contextual Semantic Similarity

Let's compare sentences with related and unrelated meanings.

In [0]:
similarity_examples = [
    "account locked after many attempts",
    "unable to login because account is locked",
    "please cancel my subscription",
    "internet connection keeps dropping",
]

In [0]:
#Generate embeddings:
similarity_embeddings = (
    embedding_model.encode(
        similarity_examples,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
)

In [0]:
#Calculate cosine similarity:
similarity_matrix = (
    cosine_similarity(
        similarity_embeddings
    )
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=similarity_examples,
    columns=similarity_examples,
)

display(similarity_df)

We would generally expect semantically related login/account-lock sentences to have relatively high similarity.

Exact values depend on the pretrained model.

##### 45. Why This Worked Better Than TF-IDF

TF-IDF learned its vocabulary only from our local training split.

If an important test word was unseen during training, that word could not
receive a TF-IDF feature.

The pretrained MiniLM Transformer had already learned language
representations from much larger external training data.

Therefore a ticket such as: "account locked after many attempts"

can still receive a meaningful semantic representation even though words
such as `locked` and `attempts` were absent from our tiny local TF-IDF
training vocabulary.

##### 46. Pretraining


##### Pretraining

A Transformer is initially trained before our support-ticket project.

Conceptually:

``` text

Large external training corpus
↓
Transformer
↓
training objective
↓
loss
↓
backpropagation
↓
gradients
↓
optimizer
↓
millions of parameter updates
↓
pretrained Transformer

```

Learned parameters can include:

- token embeddings
- position-related parameters
- attention projections
- W_Q
- W_K
- W_V
- feed-forward weights
- normalization parameters etc.

When we download the pretrained model, we reuse those learned parameters.

##### 47. Feature Extraction



This is what we did in Notebook 08.

``` text

Pretrained MiniLM
       ↓
generate sentence embeddings
       ↓
384 fixed features
       ↓
Logistic Regression

```

MiniLM served as a: pretrained feature extractor

Our Logistic Regression learned the ticket categories.

##### 48. Neural Network on Frozen Embeddings



Notebook 09 used:

``` text

Pretrained MiniLM
       ↓
384-dimensional embeddings
       ↓
Neural Network
384 → 64 → 32 → 4
       ↓
ticket category

```

Again, we generated embeddings first and trained a separate classifier.

##### 49. Fine-Tuning

Transformer fine-tuning is different.

Conceptually:

``` text

Pretrained Transformer
        ↓
classification head
        ↓
4 logits
        ↓
CrossEntropyLoss
        ↓
backpropagation
        ↓
update trainable parameters
        ↓
task-adapted Transformer

```

Instead of merely consuming frozen sentence embeddings, we can train the Transformer architecture directly for our ticket-classification task.

This is what Notebook 11 will explore.

##### 50. Encoder vs Decoder Transformers

##### Encoder-style

Encoder-style Transformers are designed primarily for producing rich representations of the available input.

Conceptually: account ←→ locked ←→ after ←→ attempts

Tokens can use context across the input sequence.

Typical tasks:

- classification
- semantic similarity
- embeddings
- named entity recognition

MiniLM/BERT-style architectures belong to this general family.

##### Decoder-style

Decoder-style Transformers are commonly used for autoregressive generation.

Conceptually:

``` text

"The customer cannot login because"
                    ↓
                   the

"The customer cannot login because the"
                    ↓
                 account

"The customer cannot login because the account"
                    ↓
                    is

```
During next-token generation, future tokens must not be visible.

Therefore decoder-style generation uses causal attention.

GPT-style models belong to this general family.

##### 51. Attention Head vs Classification Head



These two terms use the same word but refer to different things.

ATTENTION HEAD

``` text

Q/K/V
 ↓
self-attention calculation
 ↓
attention output

```
versus:

CLASSIFICATION HEAD

``` text

Transformer representation
       ↓
prediction layer
       ↓
class logits

```

For our task:

- Billing
- Cancellation
- Login
- Technical

A Transformer classifier ultimately produces four class logits.

##### 52. Transformer Learning Connection


Transformers introduce sophisticated architecture, but the underlying
learning process is still based on the neural-network concepts already
covered earlier.

``` text

Forward propagation
       ↓
Loss
       ↓
Backpropagation
       ↓
Gradients
       ↓
Optimizer
       ↓
Parameter updates

```
Therefore:

``` text

Neural Network foundations
        ↓
make Transformer training
much easier to understand

```

##### 53. Complete Conceptual Flow


``` text

RAW TEXT
"account locked after many attempts"

            ↓

TOKENIZER

            ↓

TOKENS / SUBWORDS

            ↓

TOKEN IDs

            ↓

TOKEN EMBEDDINGS
"What token is this?"

            +

POSITION INFORMATION
"Where is this token?"

            ↓

POSITION-AWARE REPRESENTATIONS

            ↓

Q = XW_Q
K = XW_K
V = XW_V

            ↓

QKᵀ

            ↓

SCALED ATTENTION SCORES

            ↓

SOFTMAX

            ↓

ATTENTION WEIGHTS

            ↓

WEIGHTED VALUE VECTORS

            ↓

SELF-ATTENTION OUTPUT

            ↓

MULTI-HEAD ATTENTION

            ↓

RESIDUAL + NORMALIZATION

            ↓

FEED-FORWARD NEURAL NETWORK

            ↓

RESIDUAL + NORMALIZATION

            ↓

REPEAT TRANSFORMER LAYERS

            ↓

CONTEXTUAL TOKEN REPRESENTATIONS

            ↓

POOLING

            ↓

384-DIMENSIONAL SENTENCE EMBEDDING

```

##### 54. Verification Checks

In [0]:
assert (
    embedding_model
    .get_sentence_embedding_dimension()
    == sentence_embeddings.shape[1]
)

assert (
    encoded_batch[
        "input_ids"
    ].shape
    ==
    encoded_batch[
        "attention_mask"
    ].shape
)

assert (
    last_hidden_state.shape[-1]
    ==
    config.hidden_size
)

assert (
    sentence_embeddings.shape[0]
    ==
    len(variable_length_tickets)
)

print(
    "All Transformer foundation "
    "verification checks passed."
)

##### 55. Key Learnings

## Key Learnings

1. Transformers do not operate directly on raw text. Text is first
   tokenized into tokens/subwords and converted to token IDs.

2. Token IDs are identifiers, not semantic numerical values.

3. Token IDs are mapped to learned token embeddings.

4. Position information allows the model to distinguish token order.

5. Token embedding answers primarily "what token is this?", while
   positional information contributes "where is it?".

6. Self-attention allows each token representation to incorporate
   information from other relevant tokens in the sequence.

7. Query, Key, and Value vectors are calculated from the current input
   using learned W_Q, W_K, and W_V projection parameters.

8. Raw attention scores are normalized with Softmax into attention
   weights.

9. Attention weights sum to 1 across the relevant allowed positions,
   but they are not class probabilities.

10. Weighted Value vectors create context-enriched token
    representations.

11. Multi-head attention performs several learned attention operations
    in parallel.

12. Transformer layers also contain feed-forward neural networks,
    residual connections, and normalization.

13. Transformer output is token-level:
    batch × sequence_length × hidden_dimension.

14. SentenceTransformer pooling converts multiple contextual token
    representations into one fixed-size sentence embedding.

15. For all-MiniLM-L6-v2, the sentence embedding contains 384 numerical
    features.

16. Sequence length and embedding dimension are different concepts.

17. Padding makes variable-length text compatible with batched tensor
    processing.

18. Attention masks identify real tokens versus padded positions.

19. Padding masks and causal masks solve different problems.

20. Pretraining learns general language representations before our
    project begins.

21. Feature extraction reuses pretrained representations without
    necessarily training the Transformer itself.

22. Fine-tuning adapts a pretrained Transformer to a specific downstream
    task using labeled data.

23. Encoder-style Transformers are commonly used for language
    understanding and representation.

24. Decoder-style Transformers use causal attention for autoregressive
    generation.

25. Transformers still rely on the same deep-learning foundations:
    forward propagation, loss, backpropagation, gradients, and
    optimization.

##### 56. Notebook Conclusion

## Conclusion

This notebook opened the pretrained Transformer that had previously been
used as a black-box sentence embedding model.

The project has now progressed from:

``` text

Bag of Words
↓
TF-IDF
↓
Pretrained Sentence Embeddings
↓
Neural Network Classification
↓
Transformer Internals

```

The key conceptual shift is that a Transformer does not represent a token
only from its identity.

It builds contextual token representations by allowing tokens to exchange
information through self-attention.

For the sentence:

"account locked after many attempts"

the representation of `account` can be influenced by tokens such as
`locked` and `attempts`.

Multiple attention heads allow several learned attention patterns to be
processed in parallel.

The Transformer then repeatedly refines these contextual representations
through attention and feed-forward layers.

SentenceTransformer adds pooling on top of the Transformer output to
produce one fixed-size 384-dimensional embedding for the entire ticket.

This explains what happened internally when MiniLM embeddings were used
in the previous Logistic Regression and Neural Network experiments.

##### 57. Next Notebook

##### Next — 11_transformer_ticket_classifier

The next notebook will move from Transformer representation learning to
Transformer-based classification.

Planned flow:

``` text

Support Ticket
↓
Tokenizer
↓
Input IDs + Attention Mask
↓
Pretrained Transformer
↓
Classification Head
↓
4 Class Logits
↓
CrossEntropyLoss
↓
Fine-Tuning
↓
Billing / Cancellation / Login / Technical

```

The Transformer classifier will use the same persisted train,
validation, and test splits so its results can later be compared fairly
with:

- TF-IDF + Logistic Regression
- Sentence Embeddings + Logistic Regression
- Sentence Embeddings + Neural Network